In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [17]:
player_play = pd.read_csv("data/player_play.csv")
game_tracking_2023 = pd.read_csv("data/game_tracking_2023.csv")
combine_results = pd.read_csv("data/combine_results.csv")

# Gather our plays

First we will filter based on if the play included an offensive lineman

Second we will filter out for only passing plays, as we are looking at pass protection

In [11]:
passing = player_play.loc[
    (player_play['lined_up_position'] == 'T') |
    (player_play['lined_up_position'] == 'G') |
    (player_play['lined_up_position'] == 'C'),
    ['game_id', 'play_id', 'nfl_id', 'lined_up_position', 'pass_result']
]

passing = passing[passing['pass_result'].notna()]

In [12]:
passing

,game_id,play_id,nfl_id,lined_up_position,pass_result
0,2025111610,2279,57183,T,I
8,2023091801,56,55976,T,IN
10,2025122701,63,57183,T,C
16,2023101503,3590,55973,G,C
21,2026010405,2568,57238,C,C
...,...,...,...,...,...
314167,2025092200,986,57330,G,R
314172,2025110300,3012,58214,G,I
314182,2025122811,2539,57207,G,C
314183,2025081052,373,58211,T,I


# Gather the tracking

For this example we will again be examining Darnell Wright and Paris Johnson Jr.

In [51]:
wright_plays = game_tracking_2023[game_tracking_2023['nfl_id'] == 55874]
paris_plays = game_tracking_2023[game_tracking_2023 ['nfl_id'] == 55870]

In [52]:
wright_weight = combine_results.loc[
    combine_results['nfl_id'] == 55874,
    'combine_weight'
].iloc[0]

paris_weight = combine_results.loc[
    combine_results['nfl_id'] == 55870,
    'combine_weight'
].iloc[0]

In [53]:
wright_plays['force'] = (
    wright_plays['a'] * (wright_weight / 2.20462)
).round(2)

paris_plays['force'] = (
    paris_plays['a'] * (paris_weight / 2.20462)
).round(2)

In [54]:
wright_plays = wright_plays.sort_values(
    ['game_id', 'play_id', 'time']
).copy()

In [55]:
wright_plays['row_num'] = (
    wright_plays.groupby(['game_id', 'play_id']).cumcount()
)

# identify when snap occurs
snap_rows = (
    wright_plays[wright_plays['event'] == 'ball_snap']
    [['game_id', 'play_id', 'row_num']]
    .rename(columns={'row_num': 'snap_row'})
)

wright_plays = wright_plays.merge(
    snap_rows,
    on=['game_id', 'play_id'],
    how='left'
)


In [69]:
wright_snap_3sec = wright_plays[
    wright_plays['row_num'].between(
        wright_plays['snap_row'],
        wright_plays['snap_row'] + 30
    )
].copy()

# Passing

Our focus here is on passing, and in order to find these passing plays we must identify every play where there is a pass

In [78]:
wright_passing = wright_snap_3sec.merge(
    passing[passing['nfl_id'] == 55874],
    on = ['game_id', 'play_id'],
    how = 'left'
)

In [85]:
wright_passing['seconds_after_snap'] = (
    wright_passing['row_num'] - wright_passing['snap_row']
) * 0.1

wright_passing['second'] = np.where(
    wright_passing['seconds_after_snap'] < 1, 1,
    np.where(
        wright_passing['seconds_after_snap'] < 2, 2, 3
    )
)

game_force = (
    wright_passing
    .groupby(['game_id', 'second'])['force']
    .mean()
    .reset_index(name='average_force')
)

game_force

,game_id,second,average_force
0,2023081251,1,208.940143
1,2023081251,2,224.973429
2,2023081251,3,267.587013
3,2023091008,1,227.767770
4,2023091008,2,211.989649
5,2023091008,3,253.165934
6,2023091706,1,221.274385
7,2023091706,2,203.967923
8,2023091706,3,245.093934
9,2023092411,1,216.025902


### Future updates

1. Need to filter out screen passes as blocking is not a factor here
2. Need to factor out qb_kneel plays as well
3. Shorten plays so that if there is a pass or a sack you stop counting the values